## Relabel country labels for the BMR

The country labels provided from the mortality rate VizHub download don't match the country labels on the country mask (and sometimes each other). To ensure they can be processed correctly this script relabels the countries to match the *country mask* [McDuffie et al. (2021)](https://doi.org/10.5281/zenodo.4642700).

This is run for each health outcome that the BMR has been calculated for.

In [1]:
import os
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Health outcomes ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [3]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")

In [4]:
# === Set GBD version ===
GBD_version = "GBD23"

This is a complete* dictionary of all country labels with an updated and associated new label

*as of 4 December 2025

In [5]:
# Dictionary mapping of country names to change FROM: TO
name_mapping = {
    'Arab Republic of Egypt': 'Egypt',
    'Argentine Republic': 'Argentina',
    'Bahamas': 'The Bahamas',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Bolivarian Republic of Venezuela': 'Venezuela',
    'Brunei Darussalam': 'Brunei',
    'Cabo Verde': 'Cape Verde',
    'Commonwealth of Dominica': 'Dominica',
    'Commonwealth of the Bahamas': 'The Bahamas',
    "Côte d'Ivoire": "Cote d'Ivoire",
    'Czechia': 'Czech Republic',
    "Democratic People's Republic of Korea": 'North Korea',
    'Democratic Republic of Sao Tome and Principe': 'Sao Tome and Principe',
    'Democratic Republic of Timor-Leste': 'Timor-Leste',
    'Democratic Socialist Republic of Sri Lanka': 'Sri Lanka',
    'Eastern Republic of Uruguay': 'Uruguay',
    'Eswatini': 'Swaziland',
    'Federal Democratic Republic of Ethiopia': 'Ethiopia',
    'Federal Democratic Republic of Nepal': 'Nepal',
    'Federal Republic of Germany': 'Germany',
    'Federal Republic of Nigeria': 'Nigeria',
    'Federal Republic of Somalia': 'Somalia',
    'Federative Republic of Brazil': 'Brazil',
    'French Republic': 'France',
    'Gabonese Republic': 'Gabon',
    'Gambia': 'The Gambia',
    'Grand Duchy of Luxembourg': 'Luxembourg',
    'Hashemite Kingdom of Jordan': 'Jordan',
    'Hellenic Republic': 'Greece',
    'Independent State of Papua New Guinea': 'Papua New Guinea',
    'Independent State of Samoa': 'Samoa',
    'Iran (Islamic Republic of)': 'Iran',
    'Islamic Republic of Afghanistan': 'Afghanistan',
    'Islamic Republic of Iran': 'Iran',
    'Islamic Republic of Mauritania': 'Mauritania',
    'Islamic Republic of Pakistan': 'Pakistan',
    'Kingdom of Bahrain': 'Bahrain',
    'Kingdom of Belgium': 'Belgium',
    'Kingdom of Bhutan': 'Bhutan',
    'Kingdom of Cambodia': 'Cambodia',
    'Kingdom of Denmark': 'Denmark',
    'Kingdom of Eswatini': 'Swaziland',
    'Kingdom of Lesotho': 'Lesotho',
    'Kingdom of Morocco': 'Morocco',
    'Kingdom of Norway': 'Norway',
    'Kingdom of Saudi Arabia': 'Saudi Arabia',
    'Kingdom of Spain': 'Spain',
    'Kingdom of Sweden': 'Sweden',
    'Kingdom of Thailand': 'Thailand',
    'Kingdom of Tonga': 'Tonga',
    'Kingdom of the Netherlands': 'Netherlands',
    'Kyrgyz Republic': 'Kyrgyzstan',
    "Lao People's Democratic Republic": 'Laos',
    'Lebanese Republic': 'Lebanon',
    'Micronesia (Federated States of)': 'Federated States of Micronesia',
    'North Macedonia': 'Macedonia',
    "People's Democratic Republic of Algeria": 'Algeria',
    "People's Republic of Bangladesh": 'Bangladesh',
    "People's Republic of China": 'China',
    'Plurinational State of Bolivia': 'Bolivia',
    'Portuguese Republic': 'Portugal',
    'Principality of Andorra': 'Andorra',
    'Principality of Monaco': 'Monaco',
    'Republic of Albania': 'Albania',
    'Republic of Angola': 'Angola',
    'Republic of Armenia': 'Armenia',
    'Republic of Austria': 'Austria',
    'Republic of Azerbaijan': 'Azerbaijan',
    'Republic of Belarus': 'Belarus',
    'Republic of Benin': 'Benin',
    'Republic of Botswana': 'Botswana',
    'Republic of Bulgaria': 'Bulgaria',
    'Republic of Burundi': 'Burundi',
    'Republic of Cabo Verde': 'Cape Verde',
    'Republic of Cameroon': 'Cameroon',
    'Republic of Chad': 'Chad',
    'Republic of Chile': 'Chile',
    'Republic of Colombia': 'Colombia',
    'Republic of Costa Rica': 'Costa Rica',
    'Republic of Croatia': 'Croatia',
    'Republic of Cuba': 'Cuba',
    'Republic of Cyprus': 'Cyprus',
    "Republic of Côte d'Ivoire": "Cote d'Ivoire",
    'Republic of Djibouti': 'Djibouti',
    'Republic of Ecuador': 'Ecuador',
    'Republic of El Salvador': 'El Salvador',
    'Republic of Equatorial Guinea': 'Equatorial Guinea',
    'Republic of Estonia': 'Estonia',
    'Republic of Fiji': 'Fiji',
    'Republic of Finland': 'Finland',
    'Republic of Ghana': 'Ghana',
    'Republic of Guatemala': 'Guatemala',
    'Republic of Guinea': 'Guinea',
    'Republic of Guinea-Bissau': 'Guinea-Bissau',
    'Republic of Guyana': 'Guyana',
    'Republic of Haiti': 'Haiti',
    'Republic of Honduras': 'Honduras',
    'Republic of Iceland': 'Iceland',
    'Republic of India': 'India',
    'Republic of Indonesia': 'Indonesia',
    'Republic of Iraq': 'Iraq',
    'Republic of Italy': 'Italy',
    'Republic of Kazakhstan': 'Kazakhstan',
    'Republic of Kenya': 'Kenya',
    'Republic of Kiribati': 'Kiribati',
    'Republic of Korea': 'South Korea',
    'Republic of Latvia': 'Latvia',
    'Republic of Liberia': 'Liberia',
    'Republic of Lithuania': 'Lithuania',
    'Republic of Madagascar': 'Madagascar',
    'Republic of Malawi': 'Malawi',
    'Republic of Maldives': 'Maldives',
    'Republic of Mali': 'Mali',
    'Republic of Malta': 'Malta',
    'Republic of Mauritius': 'Mauritius',
    'Republic of Moldova': 'Moldova',
    'Republic of Mozambique': 'Mozambique',
    'Republic of Namibia': 'Namibia',
    'Republic of Nauru': 'Nauru',
    'Republic of Nicaragua': 'Nicaragua',
    'Republic of Niue': 'Niue',
    'Republic of Palau': 'Palau',
    'Republic of Panama': 'Panama',
    'Republic of Paraguay': 'Paraguay',
    'Republic of Peru': 'Peru',
    'Republic of Poland': 'Poland',
    'Republic of Rwanda': 'Rwanda',
    'Republic of San Marino': 'San Marino',
    'Republic of Senegal': 'Senegal',
    'Republic of Serbia': 'Serbia',
    'Republic of Seychelles': 'Seychelles',
    'Republic of Sierra Leone': 'Sierra Leone',
    'Republic of Singapore': 'Singapore',
    'Republic of Slovenia': 'Slovenia',
    'Republic of South Africa': 'South Africa',
    'Republic of South Sudan': 'South Sudan',
    'Republic of Sudan': 'Sudan',
    'Republic of Suriname': 'Suriname',
    'Republic of Tajikistan': 'Tajikistan',
    'Republic of Trinidad and Tobago': 'Trinidad and Tobago',
    'Republic of Tunisia': 'Tunisia',
    'Republic of Turkey': 'Turkey',
    'Republic of Uganda': 'Uganda',
    'Republic of Uzbekistan': 'Uzbekistan',
    'Republic of Vanuatu': 'Vanuatu',
    'Republic of Yemen': 'Yemen',
    'Republic of Zambia': 'Zambia',
    'Republic of Zimbabwe': 'Zimbabwe',
    'Republic of the Congo': 'Congo',
    'Republic of the Gambia': 'The Gambia',
    'Republic of the Marshall Islands': 'Marshall Islands',
    'Republic of the Niger': 'Niger',
    'Republic of the Philippines': 'Philippines',
    'Republic of the Union of Myanmar': 'Myanmar',
    'Slovak Republic': 'Slovakia',
    'Socialist Republic of Viet Nam': 'Vietnam',
    'State of Eritrea': 'Eritrea',
    'State of Israel': 'Israel',
    'State of Kuwait': 'Kuwait',
    'State of Libya': 'Libya',
    'State of Qatar': 'Qatar',
    'Sultanate of Oman': 'Oman',
    'Swiss Confederation': 'Switzerland',
    'Syrian Arab Republic': 'Syria',
    'Taiwan (Province of China)': 'Taiwan',
    'Togolese Republic': 'Togo',
    'Union of the Comoros': 'Comoros',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'United Mexican States': 'Mexico',
    'United Republic of Tanzania': 'Tanzania',
    'United States Virgin Islands': 'Virgin Islands, U.S.',
    'United States of America': 'United States',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Viet Nam': 'Vietnam'
}

In [6]:
# Load the country masks
masks_file = "GBD_Country_Masks_0.10.nc"
masks_path = os.path.join(MASKS_DIR, masks_file)
masks = xr.open_dataarray(masks_path)

for health_VAR in health_vars:
    print(f"Processing health outcome {health_VAR}")

    bmr_file = f"{GBD_version}_BMR_Country_{health_VAR}_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    bmr = xr.open_dataarray(bmr_path)

    # Change BMR country names
    bmr = bmr.assign_coords(
        country=[name_mapping.get(c, c) for c in bmr["country"].values]
    )

    # Save output
    out_file = f"{GBD_version}_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
    out_path = os.path.join(BMR_DIR, out_file)
    bmr.to_netcdf(out_path)

print("All processing complete.")

Processing health outcome COPD
Processing health outcome DIABETES
Processing health outcome ISCHEMIC_HEART_DISEASE
Processing health outcome LOWER_RESPIRATORY_INFECTIONS
Processing health outcome LUNG_CANCER
Processing health outcome STROKE
Processing health outcome DEMENTIA
All processing complete.


## When you are working with new BMRs this will help identify mismatched country names

Then these can be added to the "master list" above

In [ ]:
countries_1 = set(masks["country"].values)  # Country labels we want
countries_2 = set(bmr["country"].values)  # Country labels we need to change

In [6]:
# Show list of country names we will change FROM
mismatched = countries_2 - countries_1
print("Mismatched country names:", sorted(mismatched))

Mismatched country names: ['Arab Republic of Egypt', 'Argentine Republic', 'Bolivarian Republic of Venezuela', 'Brunei Darussalam', 'Commonwealth of Dominica', 'Commonwealth of the Bahamas', "Democratic People's Republic of Korea", 'Democratic Republic of Sao Tome and Principe', 'Democratic Republic of Timor-Leste', 'Democratic Socialist Republic of Sri Lanka', 'Eastern Republic of Uruguay', 'Federal Democratic Republic of Ethiopia', 'Federal Democratic Republic of Nepal', 'Federal Republic of Germany', 'Federal Republic of Nigeria', 'Federal Republic of Somalia', 'Federative Republic of Brazil', 'French Republic', 'Gabonese Republic', 'Grand Duchy of Luxembourg', 'Hashemite Kingdom of Jordan', 'Hellenic Republic', 'Independent State of Papua New Guinea', 'Independent State of Samoa', 'Islamic Republic of Afghanistan', 'Islamic Republic of Iran', 'Islamic Republic of Mauritania', 'Islamic Republic of Pakistan', 'Kingdom of Bahrain', 'Kingdom of Belgium', 'Kingdom of Bhutan', 'Kingdom o

In [7]:
# Show list of country names we will change TO
mismatched = countries_1 - countries_2
print("Mismatched country names:", sorted(mismatched))

Mismatched country names: ['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Argentina', 'Armenia', 'Austria', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Belarus', 'Belgium', 'Benin', 'Bhutan', 'Bolivia', 'Botswana', 'Brazil', 'Brunei', 'Bulgaria', 'Burundi', 'Cambodia', 'Cameroon', 'Cape Verde', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Cuba', 'Cyprus', 'Denmark', 'Djibouti', 'Dominica', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon', 'Germany', 'Ghana', 'Greece', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Honduras', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Italy', 'Jordan', 'Kazakhstan', 'Kenya', 'Kiribati', 'Kuwait', 'Kyrgyzstan', 'Laos', 'Latvia', 'Lebanon', 'Lesotho', 'Liberia', 'Libya', 'Lithuania', 'Luxembourg', 'Macedonia', 'Madagascar', 'Malawi', 'Maldives', 'Mali', 'Malta', 'Marshall Islands', 'Mau